<a href="https://colab.research.google.com/github/Yusra7947/AI-Annual-Report-Consultant/blob/main/RELIANCE_ANNUAL_REPORT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-google-genai langchain-community pypdf faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [ ]:
import os
import logging
from typing import List, Dict, Any

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import google.colab.userdata

# Standard dev logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class FinancialReportEngine:
    """
    RAG-based consultant for analyzing annual reports.
    Optimized for high-density financial PDFs.
    """
    def __init__(self, pdf_path: str, model_id: str = 'gemini-flash-latest'):
        self.pdf_path = pdf_path
        self.model_id = model_id
        self._check_env()
        self.qa_chain = self._build_pipeline()

    def _check_env(self):
        key = google.colab.userdata.get("GEMINI_API_KEY")
        if not key:
            raise EnvironmentError("Missing GOOGLE_API_KEY in Colab Secrets.")
        os.environ["GEMINI_API_KEY"] = key

    def _build_pipeline(self):
        if not os.path.exists(self.pdf_path):
            raise FileNotFoundError(f"File not found: {self.pdf_path}")

        # Ingestion logic
        logger.info("Initializing document indexing...")
        loader = PyPDFLoader(self.pdf_path)
        raw_docs = loader.load()

        # Using 850/100 split for better context retention in tables
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=850, chunk_overlap=100)
        split_docs = text_splitter.split_documents(raw_docs)

        # Vector Store (Local FAISS)
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        vectorstore = FAISS.from_documents(split_docs, embeddings)
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

        # LLM & Prompt Engineering
        llm = ChatGoogleGenerativeAI(model=self.model_id, temperature=0.1, max_output_tokens=8200)

        template = """
        SYSTEM: You are a Corporate Strategy Analyst.
        Provide a concise response based on the excerpts below.
        If the data isn't present, state 'Information not found in report.'

        EXCERPTS:
        {context}

        USER QUERY: {question}
        """

        prompt = ChatPromptTemplate.from_template(template)

        # Parallel processing to return both answer and source docs
        return RunnableParallel({
            "context": retriever,
            "question": RunnablePassthrough()
        }) | {
            "answer": (lambda x: {"context": self._format_context(x["context"]), "question": x["question"]}) | prompt | llm | StrOutputParser(),
            "sources": lambda x: x["context"]
        }

    @staticmethod
    def _format_context(docs: List):
        return "\n\n".join([f"[Page {d.metadata.get('page', 'N/A')}]: {d.page_content}" for d in docs])

    def query(self, text: str) -> Dict[str, Any]:
        return self.qa_chain.invoke(text)

def main():
    # Targeted path for the Reliance FY25 report
    DATA_SOURCE = "/content/drive/MyDrive/Report-2024-25.pdf"

    try:
        engine = FinancialReportEngine(DATA_SOURCE)
        print("\n--- ANALYST TERMINAL READY ---")

        while True:
            user_q = input("\nEnter Financial Query (or 'q' to quit): ").strip()
            if user_q.lower() in ['q', 'exit', 'quit']:
                break

            if not user_q: continue

            print("Processing...")
            output = engine.query(user_q)

            print("\n" + "="*75)
            print(f" QUERY   >> {user_q}")
            print("-" * 75)

            print(f"\nRESULT:\n{output['answer']}")

            # Show sources for professional transparency
            pages = sorted(list(set([str(d.metadata.get('page', '?')) for d in output['sources']])))
            print(f"\nSOURCES: Verified against page(s): {', '.join(pages)}")

    except Exception as error:
        logger.error(f"Execution failed: {error}")

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- ANALYST TERMINAL READY ---

Enter Financial Query (or 'q' to quit): Provide a Markdown table comparing FY 2024-25 vs FY 2023-24 for: 1. Dividend Per Share, 2. Total Dividend Payout (₹ crore), and 3. Consolidated Net Profit. Then, calculate the Payout Ratio for both years.
Processing...

 QUERY   >> Provide a Markdown table comparing FY 2024-25 vs FY 2023-24 for: 1. Dividend Per Share, 2. Total Dividend Payout (₹ crore), and 3. Consolidated Net Profit. Then, calculate the Payout Ratio for both years.
---------------------------------------------------------------------------

RESULT:
Based on the excerpts provided, here is the comparison between FY 2024-25 and FY 2023-24:

### **Financial Comparison Table**

| Particulars | FY 2024-25 | FY 2023-24 |
| :--- | :--- | :--- |
| **1. Dividend Per Share (₹)** | 5.50* | 10.00 |
| **2. Total Dividend Payout (₹ crore)** | 7,443 | 6,766** |
| **3. Consolidated Net Profit (₹ crore)** | 81,309 | Information not found in report |
| **Payout Rat